# L63 profile likelihood vs sampling rate (`rho` unknown)

This notebook studies how the observation sampling rate affects the profile likelihood of the unknown Lorenz-63 parameter `rho`, while keeping all other model components fixed as in the tutorials/deep-dives.

We use `ContinuousTimeEnKF` to approximate the marginal log-likelihood for each fixed `rho` value.

## Setup

- Dynamical model: continuous-time L63 SDE
- Unknown parameter: `rho`
- True value used for synthetic data: `rho_true = 28.0`
- Time horizon: `T = 20`
- Sampling rates (seconds): `[1e-3, 1e-2, 1e-1, 1.0]`

In [1]:
import time
import numpy as np
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
from jax import grad, vmap

import numpyro
import numpyro.distributions as dist
from numpyro.infer import Predictive

import dynestyx as dsx
from dynestyx import DynamicalModel, Filter, SDESimulator
from dynestyx.models import ContinuousTimeStateEvolution, LinearGaussianObservation
from dynestyx.inference.filters import ContinuousTimeEnKFConfig

In [2]:
def l63_model(obs_times=None, obs_values=None):
    # rho is the only unknown parameter in this study.
    rho = numpyro.sample('rho', dist.Uniform(10.0, 40.0))

    state_dim = 3
    dynamics = DynamicalModel(
        initial_condition=dist.MultivariateNormal(
            loc=jnp.zeros(state_dim),
            covariance_matrix=(20.0**2) * jnp.eye(state_dim),
        ),
        state_evolution=ContinuousTimeStateEvolution(
            drift=lambda x, u, t: jnp.array([
                10.0 * (x[1] - x[0]),
                x[0] * (rho - x[2]) - x[1],
                x[0] * x[1] - (8.0 / 3.0) * x[2],
            ]),
            diffusion_coefficient=lambda x, u, t: jnp.eye(state_dim),
        ),
        observation_model=LinearGaussianObservation(
            H=jnp.eye(state_dim),
            R=jnp.eye(state_dim),
        ),
    )
    return dsx.sample('f', dynamics, obs_times=obs_times, obs_values=obs_values)

## Generate one high-rate synthetic dataset and downsample

In [ ]:
T = 200.0
sample_rates = [1e-3, 1e-2, 1e-1, 1.0]
base_dt = 1e-3
rho_true = 28.0

base_times = jnp.arange(0.0, T, base_dt)

key = jr.PRNGKey(0)
predictive = Predictive(
    l63_model,
    params={'rho': jnp.array(rho_true)},
    num_samples=1,
    exclude_deterministic=False,
)

from diffrax import DirectAdjoint
with SDESimulator(unsafe_brownian_path=True, adjoint=DirectAdjoint()):
    sim = predictive(key, obs_times=base_times)

base_obs = sim['observations'].squeeze(0)
base_states = sim['states'].squeeze(0)

print('Base observation shape:', base_obs.shape)
print('Base time points:', base_times.shape[0])

plt.figure(figsize=(9, 3.5))
for i, name in enumerate(['x', 'y', 'z']):
    plt.plot(np.asarray(base_times), np.asarray(base_obs)[:, i], lw=1, label=name)
plt.title('Synthetic L63 observations at dt=1e-3')
plt.xlabel('time')
plt.ylabel('state')
plt.legend(ncol=3, frameon=False)
plt.tight_layout()
plt.show()

TypeError: DirectAdjoint.loop() missing 1 required positional argument: 'self'

In [ ]:
def downsample_from_base(dt):
    step = int(round(dt / base_dt))
    if not np.isclose(step * base_dt, dt):
        raise ValueError(f'dt={dt} is not an integer multiple of base_dt={base_dt}')
    return base_times[::step], base_obs[::step]

for dt in sample_rates:
    t_dt, y_dt = downsample_from_base(dt)
    print(f'dt={dt:<6g} -> n_obs={t_dt.shape[0]}')

## Profile likelihood helper (`ContinuousTimeEnKF`)
The deterministic site `f_marginal_loglik` is added by `Filter` backends.

In [ ]:
def mll_at_rho(rho, obs_times, obs_values, filter_config, key):
    def filtered_model(obs_times=None, obs_values=None):
        with Filter(filter_config=filter_config):
            return l63_model(obs_times=obs_times, obs_values=obs_values)

    conditioned = numpyro.handlers.condition(
        filtered_model,
        data={'rho': rho},
    )
    tr = numpyro.handlers.trace(numpyro.handlers.seed(conditioned, key)).get_trace(
        obs_times=obs_times,
        obs_values=obs_values,
    )
    return tr['f_marginal_loglik']['value']


def profile_and_grad_for_dt(dt, rho_grid, n_particles=25, seed=123):
    obs_times, obs_values = downsample_from_base(dt)
    cfg = ContinuousTimeEnKFConfig(n_particles=n_particles, crn_seed=jr.PRNGKey(seed))
    key = jr.PRNGKey(seed)

    mll_fn = lambda r: mll_at_rho(r, obs_times, obs_values, cfg, key)

    t0 = time.perf_counter()
    profile = vmap(mll_fn)(rho_grid)
    grad_profile = vmap(grad(mll_fn))(rho_grid)
    elapsed = time.perf_counter() - t0

    return np.asarray(profile), np.asarray(grad_profile), elapsed, obs_times.shape[0]

## Compute profiles for each sample rate

In [ ]:
# Adjust this if runtime is too long.
rho_grid = jnp.linspace(10.0, 40.0, 30)

profiles = {}
grad_profiles = {}
meta = {}

for i, dt in enumerate(sample_rates):
    prof, grad_prof, elapsed, n_obs = profile_and_grad_for_dt(
        dt=dt,
        rho_grid=rho_grid,
        n_particles=25,
        seed=100 + i,
    )
    profiles[dt] = prof
    grad_profiles[dt] = grad_prof
    meta[dt] = {'elapsed_s': elapsed, 'n_obs': int(n_obs)}
    print(f'dt={dt:<6g} | n_obs={n_obs:<6d} | elapsed={elapsed:6.2f}s')

## Plot profile likelihoods colored by sample rate

In [ ]:
plt.figure(figsize=(8.5, 4.5))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(sample_rates)))

for c, dt in zip(colors, sample_rates):
    y = profiles[dt]
    plt.plot(
        np.asarray(rho_grid),
        y,
        marker='o',
        ms=3.5,
        lw=1.7,
        color=c,
        label=f'dt={dt:g} (n={meta[dt]["n_obs"]})',
    )
    i_star = int(np.argmax(y))
    rho_hat = float(rho_grid[i_star])
    y_hat = float(y[i_star])


    plt.scatter([rho_hat], [y_hat], marker='d',color=c, s=70, edgecolors='black', linewidth=0.6, zorder=5)

plt.axvline(rho_true, color='k', ls='--', lw=1.2, label=r'$\rho_{true}$')
plt.xlabel(r'$\rho$')
plt.ylabel(r'$\log p(y_{1:T} \mid \rho)$ (EnKF approx.)')
plt.yscale('symlog')
plt.ylim(top=-50)
plt.title('L63 profile likelihood vs sampling rate (T=20, ContinuousTimeEnKF)')
plt.grid(alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()

plt.show()

In [ ]:
# Gradient of MLL with respect to rho across sampling rates.
plt.figure(figsize=(8.5, 4.5))
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(sample_rates)))

for c, dt in zip(colors, sample_rates):
    dy = grad_profiles[dt]
    plt.plot(
        np.asarray(rho_grid),
        dy,
        marker='o',
        ms=3.2,
        lw=1.6,
        color=c,
        label=f'dt={dt:g}',
    )

plt.axhline(0.0, color='k', ls='--', lw=1.0)
plt.axvline(rho_true, color='gray', ls=':', lw=1.1, label=r'$\rho_{true}$')
plt.yscale('symlog')
plt.xlabel(r'$\rho$')
plt.ylabel(r'$\partial \log p(y_{1:T}\mid\rho) / \partial \rho$')
plt.title('MLL gradient w.r.t. rho by sample rate (ContinuousTimeEnKF)')
plt.grid(alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()